In [ ]:
import cv2
import os
import glob
import re
import IPython.display as display
from IPython.display import Video

In [ ]:
# Define input folder containing TIFF images
input_folder = r"E:\Fatemeh\20250619-Uptake\S4"  # Change this to your actual folder path

# Set output video file (Change the path if needed)
output_video = r"E:\Fatemeh\20250619-Uptake\S4\timelapsqe.mp4"  # You can change this to another directory

if os.path.exists(input_folder):
    print("Folder found!")
else:
    print("Error: Folder not found! Check the path.")

In [ ]:
# Get all TIFF files (sorted)
image_files = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

# If no .tif files found, try searching for .tiff
if not image_files:
    image_files = sorted(glob.glob(os.path.join(input_folder, "*.tiff")))

# Check if TIFF files exist
if not image_files:
    print("❌ No TIFF images found! Check file format and extensions.")
    raise SystemExit()  # Stop execution

print(f"✅ Found {len(image_files)} TIFF files. First 5 files:")
print(image_files[:5])  # Print the first 5 found TIFF files

In [ ]:
# Frame rate (adjust as needed)
frame_rate = 30 

# Function to extract numeric time index from filenames
def extract_time_index(filename):
    match = re.search(r"t(\d+)", filename)  # Look for "t" followed by a number
    return int(match.group(1)) if match else float('inf')  # If no match, send to the end

# Sort images based on extracted time index
image_files.sort(key=extract_time_index)

# Read the first image to get frame size
first_frame = cv2.imread(image_files[0])

# Check if the image is readable
if first_frame is None:
    print("❌ Error: Unable to read the first image. Check if files are corrupted.")
    raise SystemExit()

height, width, _ = first_frame.shape

# Define video codec and create VideoWriter
fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # Try "XVID" or "MJPG" if mp4v doesn't work
video = cv2.VideoWriter(output_video, fourcc, frame_rate, (width, height))

# Initialize frame counter
frame_count = 0

# Process images and add them to video (Skip every 5th frame)
for i, img_file in enumerate(image_files):
    if i % 50 == 0:  # Print progress every 50 frames
        print(f"📷 Processing frame {i}/{len(image_files)}")

    if i % 1 != 0:  # Skip frames (only process every 5th image)
        continue

    frame = cv2.imread(img_file)
    if frame is not None:
        video.write(frame)
        frame_count += 1  # Count valid frames
    else:
        print(f"⚠️ Warning: Skipping unreadable file {img_file}")
        
        
# Release video resources
video.release()
cv2.destroyAllWindows()

# Check if video has frames
if frame_count == 0:
    print("❌ No frames were written! The video file is empty.")
else:
    print(f"✅ Time-lapse video saved as {output_video}")
    print(f"🎥 Total frames added: {frame_count}")

    # Print the absolute path of the video file
    print("🔍 File saved at:", os.path.abspath(output_video))

    # Display video in Jupyter Notebook (if applicable)
    display.Video(output_video, embed=True)